In [ ]:
import pandas as pd
import scipy.stats as st
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
import torch
import re
import os
from dotenv import load_dotenv
from google import genai
from elevenlabs.client import ElevenLabs
# from sentence_transformers import SentenceTransformer

In [4]:
print(torch.cuda.is_available())
vul = pd.read_csv("data/vulnerabilities.csv.gz")
exploitSignals = pd.read_csv("data/exploit_signals_2024.csv.gz")
exploitSignals2 = pd.read_csv("data/exploit_signals_2025.csv.gz")
knownExploits = pd.read_csv("data/known_exploited.csv.gz")
scoreBoard = pd.read_csv("data/scoreboard_2026.csv.gz")
weaknesses = pd.read_csv("data/weakness_types.csv.gz")
ref = pd.read_csv("data/references.csv.gz")
affects = pd.read_csv(
    "data/affected_products.csv.gz",
    dtype={
        "version": str,
        "update": str,
        "version_start_including": str,
        "version_start_excluding": str,
        "version_end_including": str,
        "version_end_excluding": str,
    },
    low_memory=False,
)

True


In [5]:
# Combine 2024 and 2025 monthly signals
signals = pd.concat([exploitSignals, exploitSignals2], ignore_index=True)

# Parse KEV date_added to define your training ground truth
knownExploits["date_added"] = pd.to_datetime(knownExploits["date_added"])
kev_dates = dict(zip(knownExploits["cve_id"], knownExploits["date_added"]))

In [6]:
# Convert signal month (YYYY-MM) to month-end timestamp
signals["month_end"] = pd.to_datetime(signals["month"]) + pd.offsets.MonthEnd(1)

# Map KEV addition date
signals["kev_date"] = signals["cve_id"].map(kev_dates)

# Filter out rows where the CVE was ALREADY on KEV on or before the snapshot date
signals = signals[signals["in_kev"] == 0].copy()

# Label = 1 if added to KEV within 90 days after this snapshot month
signals["days_until_kev"] = (signals["kev_date"] - signals["month_end"]).dt.days
signals["target"] = (
    (signals["days_until_kev"] > 0) & (signals["days_until_kev"] <= 90)
).astype(int)

print("Positive samples in training window:", signals["target"].sum())

Positive samples in training window: 533


In [7]:
# Safe static columns available from the initial disclosure
safe_vul_cols = [
    "cve_id",
    "description",
    "assigner",
    "cwe_ids",
    "attack_vector",
    "attack_complexity",
    "privileges_required",
    "user_interaction",
    "cvss3_score",
]

# Keep only columns present in the dataset
available_cols = [c for c in safe_vul_cols if c in vul.columns]
vul_clean = vul[available_cols].copy()

# Merge with training signals
train_df = signals.merge(vul_clean, on="cve_id", how="left")

# Merge with 2026 test scoreboard
test_df = scoreBoard.merge(vul_clean, on="cve_id", how="left")

Run the next part if you have a powerful gpu or cpu, if not skip to the next part, since I have already made a embedded 'granite_embeddings.npy' in the folder

In [ ]:
# Load on CPU with float32 (runs much faster than bfloat16 on CPU)
model = SentenceTransformer(
    "ibm-granite/granite-embedding-125m-english",
    model_kwargs={"torch_dtype": torch.float32},
    device="cuda" if torch.cuda.is_available() else "cpu",
)

# Extract only the 72,670 scoreboard descriptions
test_cves = scoreBoard[["cve_id"]].merge(
    vul[["cve_id", "description"]], on="cve_id", how="left"
)
test_cves["description"] = test_cves["description"].fillna("None")
texts = test_cves["description"].tolist()

# Encode with large batch size
with torch.inference_mode():
    embeddings = model.encode(
        texts,
        batch_size=128,
        show_progress_bar=True,
        normalize_embeddings=True,
    )

In [ ]:
embeddings = np.load("data/granite_embeddings.npy.gz")
print("Embeddings loaded locally! Shape:", embeddings.shape)

# Verify the row count matches scoreBoard exactly (72,670)
assert embeddings.shape[0] == len(scoreBoard)

Embeddings loaded locally! Shape: (72670, 768)


In [17]:
# 1. Prepare base scoreboard dataset
test_df = scoreBoard.copy()

# Merge safe static metadata (descriptions, cwe_ids, assigner)
metadata_cols = [
    c
    for c in ["cve_id", "description", "cwe_ids", "assigner"]
    if c in vul.columns
]
test_df = test_df.merge(vul[metadata_cols], on="cve_id", how="left")
test_df["description"] = test_df["description"].fillna("")
test_df["epss"] = test_df["epss"].fillna(0.0)

# 2. Extract heuristic risk indicators from descriptions
desc = test_df["description"].str.lower()
test_df["is_rce"] = desc.str.contains(
    r"remote code execution|execute arbitrary code|command injection",
    regex=True,
).astype(int)
test_df["is_auth_bypass"] = desc.str.contains(
    r"bypass authentication|unauthenticated|improper authentication", regex=True
).astype(int)
test_df["is_perimeter"] = desc.str.contains(
    r"vpn|firewall|gateway|router|appliance|citrix|fortinet|palo alto|ivanti",
    regex=True,
).astype(int)

# 3. Public exploit presence flag
test_df["any_exploit"] = (
    (test_df["poc_github"] == 1)
    | (test_df["nuclei"] == 1)
    | (test_df["exploitdb"] == 1)
    | (test_df["metasploit"] == 1)
).astype(int)

# 4. High-risk CWE flag
high_risk_cwes = [
    "CWE-77",
    "CWE-78",
    "CWE-502",
    "CWE-287",
    "CWE-306",
    "CWE-22",
    "CWE-89",
]
test_df["high_risk_cwe"] = (
    test_df["cwe_ids"]
    .fillna("")
    .apply(lambda x: int(any(cwe in str(x) for cwe in high_risk_cwes)))
)

# 5. Semantic Profile Scoring using your Granite Embeddings
# The embeddings represent the exact rows of scoreBoard
semantic_archetype_direction = embeddings.mean(axis=0)
test_df["semantic_score"] = embeddings @ semantic_archetype_direction

# 6. Composite Threat Prioritization Score
test_df["prediction_score"] = (
    (test_df["epss"] * 4.0)
    + (test_df["any_exploit"] * 3.0)
    + (test_df["poc_github"] * 2.0)
    + (test_df["is_perimeter"] * 2.5)
    + (test_df["is_rce"] * 2.0)
    + (test_df["is_auth_bypass"] * 1.5)
    + (test_df["high_risk_cwe"] * 1.0)
    + (test_df["semantic_score"] * 1.5)
)

# 7. Rank and export top 500
top500 = test_df.sort_values(by="prediction_score", ascending=False).head(500)
top500[["cve_id"]].to_csv("top500.csv", index=False)

# Validate format: exactly 500 rows, 1 column
assert top500[["cve_id"]].shape == (500, 1), (
    f"Expected (500, 1), got {top500[['cve_id']].shape}"
)
print("Saved top500.csv successfully!")
print("Top 5 highest risk CVEs:")
print(top500[["cve_id", "prediction_score", "epss", "any_exploit"]].head())

Saved top500.csv successfully!
Top 5 highest risk CVEs:
               cve_id  prediction_score     epss  any_exploit
62040   CVE-2026-8037         15.077365  0.84793            1
10835  CVE-2026-11834         13.207862  0.01023            1
17795  CVE-2026-20251         12.446306  0.18991            1
9550   CVE-2026-10523         12.258395  0.51870            1
31762  CVE-2026-39808         12.210509  0.13099            1


In [23]:
load_dotenv()

# Pass your key directly or set it as an environment variable (GEMINI_API_KEY)
client = genai.Client(
    api_key = os.getenv("GEMINI_API_KEY")
) 

# Grab the top 5 predicted vulnerabilities
top_samples = test_df.sort_values(by="prediction_score", ascending=False).head(5)

for _, row in top_samples.iterrows():
  prompt = f"""
    You are an IBM threat intelligence engineer. Summarize why this vulnerability must be patched urgently in 2 concise sentences.
    CVE: {row['cve_id']}
    Description: {row['description']}
    EPSS Score: {row['epss']}
    Public Exploit Flag: {row['any_exploit']}
    """

  response = client.models.generate_content(
      model="gemini-3.5-flash-lite", contents=prompt
  )

  print(f"=== {row['cve_id']} Remediation Advisory ===")
  print(response.text.strip())
  print("-" * 50)

=== CVE-2026-8037 Remediation Advisory ===
CVE-2026-8037 is a critical unauthenticated OS command injection vulnerability in Progress ADC products that allows remote attackers to execute arbitrary code on the LoadMaster appliance. With an EPSS score of 0.84793 and active public exploits available, immediate patching is vital to prevent total system compromise.
--------------------------------------------------
=== CVE-2026-11834 Remediation Advisory ===
CVE-2026-11834 allows an adjacent unauthenticated attacker to achieve full device compromise and administrative control via a command injection flaw during DHCP provisioning. With public exploits already available, this vulnerability poses an immediate risk to factory-default or unconfigured TP-Link routers and must be patched urgently.
--------------------------------------------------
=== CVE-2026-20251 Remediation Advisory ===
CVE-2026-20251 allows low-privileged users to achieve Remote Code Execution via unsafe deserialization of KV

In [24]:
sub = pd.read_csv("top500.csv")
print("Shape:", sub.shape)  # Must be (500, 1)
print("Columns:", sub.columns.tolist())  # Must be ['cve_id']
print(sub.head())

Shape: (500, 1)
Columns: ['cve_id']
           cve_id
0   CVE-2026-8037
1  CVE-2026-11834
2  CVE-2026-20251
3  CVE-2026-10523
4  CVE-2026-39808
